In [5]:
!pip install -q --upgrade openai-agents

In [6]:
!pip install -q gradio openai python-dotenv requests httpx pillow

In [7]:
# Import libraries for the notebook client
import os
import requests  # For making HTTP requests
import httpx  # An alternative async-friendly HTTP client (good practice)
import json  # For handling JSON data (manifests, action responses)
from dotenv import load_dotenv
from IPython.display import display, Markdown, Image  # To display results nicely
from openai import OpenAI  # or litellm, groq, etc.

from PIL import Image
import asyncio, pathlib

# Load environment variables (needed for the Gradio apps when they run)
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

In [8]:
# Define a helper function to display markdown nicely
def print_markdown(text):
    display(Markdown(text))

In [9]:
# Let's define the URL where our Gradio MCP servers will be running
# IMPORTANT: Make sure these match the ports you use when running the server scripts!
MCP_BASE = "http://localhost:7860/gradio_api/mcp/sse"  # text tutor demo you built

# MCPServerSse: A class that sets up a server using Server-Sent Events (SSE) to serve MCP-compatible tools or agents.
# MCPServerSse allows your tool (e.g., an AI model wrapped with Gradio) to be served as an MCP tool endpoint.
# It allows Communication via SSE, which is a way to push updates from server to client (used in real-time AI applications).
from agents.mcp import MCPServerSse


# timeout: 60
# This controls how long (in seconds) the server waits for a single tool execution (i.e., an action call) before giving up.

# client_session_timeout_seconds: 180
# This sets how long an entire client session is kept alive without activity before it's considered expired or disconnected.
# If the client doesn’t send any requests or interact for 180 seconds, the session times out.

mcp_tool = MCPServerSse({
    "name": "AI Tutor",
    "url": MCP_BASE,
    "timeout": 60,
    "client_session_timeout_seconds": 180
})


In [10]:
# This code is designed to fetch and display a schema (manifest) from an MCP server
# Typically used to describe available tools on that server.

# Let's use httpx for modern async-friendly requests (though requests works fine too)
client = httpx.Client()  # Create an HTTP client instance


def fetch_schema(server_url):
    """Fetches and parses the MCP schema from a server."""

    schema_url = server_url.replace("/sse", "/schema")
    print(f"Fetching schema from: {schema_url}")

    response = client.get(schema_url, timeout = 10)  # Add a timeout
    response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)
    schema_data = response.json()
    print("Schema fetched successfully!")
    return schema_data


In [11]:
# Fetch the manifest from the AI Tutor server

# Note that the schema is used by AI agents to know how to call the AI Tutor tool programmatically, ensuring the inputs match the required format.

print("--- Fetching AI Tutor Schema ---")
tutor_schema = None

try:
    tutor_schema = fetch_schema(MCP_BASE)
except httpx.ConnectError:
    print(f"Could not connect to MCP server at: {MCP_BASE}")
    print("Make sure your Gradio MCP server is running, then retry.")
except httpx.HTTPStatusError as e:
    print(f"Server responded with HTTP error: {e.response.status_code}")
except Exception as e:
    print(f"Unexpected error while fetching schema: {e}")

if tutor_schema:
    print("\nAI Tutor Schema Contents:")
    # Pretty print the JSON manifest
    print(json.dumps(tutor_schema, indent=2))
else:
    print("Schema not available right now.")

print("\n" + "=" * 50 + "\n")  # Separator


--- Fetching AI Tutor Schema ---
Fetching schema from: http://localhost:7860/gradio_api/mcp/schema
Schema fetched successfully!

AI Tutor Schema Contents:
[
  {
    "name": "explain_concept",
    "description": "Stream an explanation of *question* at the requested *level* (1\u20115).",
    "inputSchema": {
      "type": "object",
      "properties": {
        "question": {
          "type": "string",
          "description": ""
        },
        "level": {
          "type": "number",
          "description": "",
          "default": 3
        }
      }
    },
    "meta": {
      "file_data_present": false,
      "mcp_type": "tool",
      "endpoint_name": "explain_concept"
    }
  },
  {
    "name": "summarize_text",
    "description": "Stream a summary of *text* compressed to roughly *compression_ratio* length. *compression_ratio* should be between 0.1 and 0.8.",
    "inputSchema": {
      "type": "object",
      "properties": {
        "text": {
          "type": "string",
          

In [12]:
# Let's Build the AI agent
from agents import Agent, Runner

agent = Agent(
    name = "Smart Assistant",
    instructions = """
    Context
    -------
    You are an AI assistant with access to an MCP server exposing **four streaming tools**:

    1. **explain_concept**
    Arguments: { "question": <str>, "level": <int 1‑5> }
    • Streams an explanation of any concept at the requested depth.

    2. **summarize_text**
    Arguments: { "text": <str>, "compression_ratio": <float 0.1‑0.8> }
    • Streams a concise summary ~compression_ratio × original length.

    3. **generate_flashcards**
    Arguments: { "topic": <str>, "num_cards": <int 1‑20> }
    • Streams JSON‑lines flashcards: one card per line `{ "q":…, "a":… }`.

    4. **quiz_me**
    Arguments: { "topic": <str>, "level": <int 1‑5>, "num_questions": <int 1‑15> }
    • Streams an MC‑question quiz, then an ANSWER KEY section.

    Objective
    ---------
    Help users learn by:
    • Explaining concepts at the depth they request.
    • Summarising long passages.
    • Generating flashcards for self‑study.
    • Quizzing them interactively.

    How to respond
    --------------
    • For each user request, decide which tool (if any) fulfils it best.
    • Call the tool via MCP by returning *only* the JSON with `"tool"` and `"arguments"` (no extra text).
    • If a follow‑up conversation is needed (e.g., clarification), ask the user first.
    • If no tool fits, answer directly in plain language.

    Examples
    --------
    User: “Explain quantum tunnelling like I’m 10.”
    → Call `explain_concept` with { "question": "quantum tunnelling", "level": 2 }

    User: “Summarise this article to 20 %.” + <article text>
    → Call `summarize_text` with { "text": "...", "compression_ratio": 0.2 }

    Chat capability
    ---------------
    After each tool call completes (streaming back to the user), remain in the chat loop ready for the next user turn.
    """,
    model = "gpt-4o-mini",
    mcp_servers = [mcp_tool],
)

In [13]:
# This code snippet is implementing a conversational loop with an AI agent that uses MCP tools.

# Opens a connection with an MCP tool.
# This lets our AI agent interact with an external tool (e.g., image generator, calculator, etc.) over a standard protocol using SSE (Server-Sent Events).
await mcp_tool.connect()  # open SSE channels

result = None
while True:
    user_input = input("User: ")
    if user_input.lower() in {"exit", "quit"}:
        break

    # If there was a previous interaction (result is not None), it appends the new user message to the past messages (maintaining conversation context).
    if result is not None:
        new_input = result.to_input_list() + [{"role": "user", "content": user_input}]
    else:
        new_input = [{"role": "user", "content": user_input}]
    print("\nUser Input:")
    print_markdown(user_input)


   # This is the core AI agent execution step. It runs your agent with the new_input.

    result = await Runner.run(starting_agent = agent, input = new_input)
    print("\nAssistant:")
    print_markdown(result.final_output)


User Input:


newton first law


User Input:


newton first law

Error invoking MCP tool explain_concept on server 'sse: http://localhost:7860/gradio_api/mcp/sse': Timed out while waiting for response to ClientRequest. Waited 5.0 seconds.
Error invoking MCP tool explain_concept on server 'sse: http://localhost:7860/gradio_api/mcp/sse': Timed out while waiting for response to ClientRequest. Waited 5.0 seconds.


...


Assistant:


It seems like your messages are not coming through. If you have a question or need assistance, please try sending it again!


User Input:



Assistant:


I haven't received any clear messages from you. If you have a question or need assistance, please feel free to type it out!

In [15]:
# Let's view the list of tools that have been called
if "result" not in globals() or result is None:
    print("No agent result is available yet. Run the chat cell first.")
else:
    for i in result.to_input_list():
        if "arguments" in i:
            print("Tool: ", i.get("name", "unknown"))
            print("Arguments: ", i["arguments"])

Tool:  explain_concept
Arguments:  {"question":"Newton's first law of motion","level":3}
Tool:  explain_concept
Arguments:  {"question":"Newton's first law of motion","level":3}
